In [ ]:
import kagglehub
import os



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
# Task 1: Write your code here:

df_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:

# Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here:

# Display dataset information using info()

df.info()

In [ ]:
# Task 4: Write your code here:

# Show statistical description using describe()
df.describe()

In [ ]:
# Task 1: Write your code here:

# Calculate missing percentages
missing_percentages = df.isnull().sum() / len(df) * 100

# Identify columns to drop (more than 70% missing)
columns_to_drop = missing_percentages[missing_percentages > 70].index

# Drop columns from the DataFrame
df_cleaned = df.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns with more than 70% missing values.")
print("Remaining columns after dropping high-missing-value columns:", df_cleaned.shape[1])

# Identify numerical columns with remaining missing values and impute with median
numerical_cols_with_missing = df_cleaned.select_dtypes(include=['number']).columns[df_cleaned.select_dtypes(include=['number']).isnull().any()].tolist()

for col in numerical_cols_with_missing:
    median_val = df_cleaned[col].median()
    df_cleaned[col] = df_cleaned[col].fillna(median_val) # Assign back to avoid SettingWithCopyWarning

print(f"Imputed missing values in {len(numerical_cols_with_missing)} numerical columns with their median.")

# Verify that there are no more missing values in numerical columns
print("\nMissing values after imputation:")
print(df_cleaned.select_dtypes(include=['number']).isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:

initial_rows = df_cleaned.shape[0]
duplicate_rows = df_cleaned.duplicated().sum()

if duplicate_rows > 0:
    df_cleaned.drop_duplicates(inplace=True)
    print(f"Found and removed {duplicate_rows} duplicate rows.")
else:
    print("No duplicate rows found.")

rows_after_deduplication = df_cleaned.shape[0]
print(f"Number of rows before deduplication: {initial_rows}")
print(f"Number of rows after deduplication: {rows_after_deduplication}")

In [ ]:
# Task 3: Write your code here:

categorical_cols = df_cleaned.select_dtypes(include='object').columns

if len(categorical_cols) > 0:
    print(f"Found {len(categorical_cols)} categorical columns: {categorical_cols.tolist()}")
    # Apply one-hot encoding if needed. 'drop_first=True' prevents multicollinearity.
    df_encoded = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True)
    df_cleaned = df_encoded # Update df_cleaned with encoded data
    print("Categorical columns one-hot encoded.")
    print(f"Shape after encoding: {df_cleaned.shape}")
else:
    print("No 'object' type categorical columns found. All features are numerical or already handled.")


In [ ]:
from sklearn.preprocessing import StandardScaler

# Task 4: Write your code here:

# Create a copy to avoid modifying the original df_cleaned directly and prevent SettingWithCopyWarning
df_for_scaling = df_cleaned.copy()

# Separate features (X) and target (y)
X = df_for_scaling.drop('Target', axis=1)
y = df_for_scaling['Target']

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to all features in X
X_scaled = scaler.fit_transform(X)

# Convert scaled features back to a DataFrame, preserving column names and index
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

# Combine scaled features with the target variable to form the final processed DataFrame
df_processed = X_scaled_df.copy()
df_processed['Target'] = y

print("Numerical features scaled using StandardScaler.")
print(f"Shape of processed DataFrame after scaling: {df_processed.shape}")
print(f"First 5 rows of scaled features (including target):\n{df_processed.head()}")

In [ ]:
# Task 5: Write your code here:

target_distribution = df_processed['Target'].value_counts(normalize=True)
print("Target Variable Distribution:")
print(target_distribution)

# Determine if the target is imbalanced. A common heuristic is if the minority class is < 20-30%.
imbalance_threshold = 0.20 # Define a threshold for what constitutes significant imbalance
if target_distribution.min() < imbalance_threshold:
    print(f"Conclusion: The target variable is imbalanced. The minority class (label {target_distribution.idxmin()}) accounts for {target_distribution.min():.2%} of the data.")
else:
    print("Conclusion: The target variable is not significantly imbalanced.")

In [ ]:
!pip install catboost

In [ ]:
# Task 1: Write your code here:

import numpy as np
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

# X and y are already defined from the previous scaling step using df_processed
X = df_processed.drop('Target', axis=1)
y = df_processed['Target']

print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

print("\n--- Part 3: Modeling ---")

# Task 1: Split the dataset into features (X) and target (y)
# X and y are already defined from the previous scaling step using df_processed
X = df_processed.drop('Target', axis=1)
y = df_processed['Target']

print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")

# Task 2,3,4,5: Write your code here:

# Use the correct split: StratifiedKFold
# StratifiedKFold is appropriate due to the target imbalance identified in Part 2.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# List to store F1-scores from each fold
f1_scores = []

print("\nStarting Stratified K-Fold Cross-Validation...")
for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train a CatBoostClassifier model
    # Using default parameters for simplicity; can be tuned for better performance
    model = CatBoostClassifier(
        iterations=100,  # Number of boosting rounds
        learning_rate=0.1,
        depth=6,
        loss_function='Logloss',
        eval_metric='F1', # Evaluate using F1-score
        random_seed=42,
        verbose=0,       # Suppress verbose output
        early_stopping_rounds=10 # Stop if F1 doesn't improve for 10 rounds
    )

    print(f"\nTraining Fold {fold+1}...")
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=10, verbose=0)

    # Evaluate using the appropriate metric (F1 Score)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)
    print(f"Fold {fold+1} F1-Score: {f1:.4f}")

# Print the averaged score across all folds
average_f1_score = np.mean(f1_scores)
print(f"\nAverage F1-Score across all folds: {average_f1_score:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Ensure X (features) and model are available from Part 3
# If this cell is run independently, X and model might not be in scope.
# Assuming the previous cells have been run, X and model are available.

# Get feature importances from the trained model
feature_importances = model.get_feature_importance()

# Create a Series with feature names and their importances
feature_names = X.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})

# Sort the features by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
# Task 1: Write your code here:

# Plot feature importance from your trained model
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(20))
plt.title('Top 20 Feature Importances from CatBoost Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

golden_feature = importance_df.iloc[0]['Feature']
golden_feature_importance = importance_df.iloc[0]['Importance']

print(f"\nThe 'Golden Feature' (most important predictor) is: '{golden_feature}' with an importance of {golden_feature_importance:.4f}")

In [ ]:
# Retrain with Golden Feature Only
import numpy as np
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

print("\n--- Part 5: Bonus - Retrain with Golden Feature Only ---")

# 1. Create new X with only the golden feature
X_golden = df_processed[[golden_feature]]
y = df_processed['Target'] # Target remains the same

print(f"Shape of X_golden (single golden feature): {X_golden.shape}")

# 2. Run the same KFold loop with this single feature
skf_golden = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
golden_f1_scores = []

print("\nStarting Stratified K-Fold Cross-Validation with Golden Feature...")
for fold, (train_index, test_index) in enumerate(skf_golden.split(X_golden, y)):
    X_train_golden, X_test_golden = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train_golden, y_test_golden = y.iloc[train_index], y.iloc[test_index]

    # Train a CatBoostClassifier model with only the golden feature
    model_golden = CatBoostClassifier(
        iterations=100,  # Number of boosting rounds
        learning_rate=0.1,
        depth=6,
        loss_function='Logloss',
        eval_metric='F1', # Evaluate using F1-score
        random_seed=42,
        verbose=0,       # Suppress verbose output
        early_stopping_rounds=10 # Stop if F1 doesn't improve for 10 rounds
    )

    print(f"Training Fold {fold+1} (Golden Feature Only)...")
    model_golden.fit(X_train_golden, y_train_golden, eval_set=(X_test_golden, y_test_golden), early_stopping_rounds=10, verbose=0)

    # Evaluate using F1 Score
    y_pred_golden = model_golden.predict(X_test_golden)
    f1_golden = f1_score(y_test_golden, y_pred_golden)
    golden_f1_scores.append(f1_golden)
    print(f"Fold {fold+1} F1-Score (Golden Feature Only): {f1_golden:.4f}")

# Print the averaged score across all folds for the golden feature model
average_golden_f1_score = np.mean(golden_f1_scores)
print(f"\nAverage F1-Score across all folds (Golden Feature Only): {average_golden_f1_score:.4f}")

# 3. Print and compare the accuracy (F1-score) with the full model
# Assuming 'average_f1_score' from the full model is available from Part 3
print(f"\n--- Comparison with Full Model ---")
print(f"Full Model Average F1-Score: {average_f1_score:.4f}")
print(f"Golden Feature Only Model Average F1-Score: {average_golden_f1_score:.4f}")

if average_golden_f1_score > average_f1_score:
    print("Conclusion: The Golden Feature model performed better than the Full Model.")
elif average_golden_f1_score == average_f1_score:
    print("Conclusion: The Golden Feature model performed equally to the Full Model.")
else:
    print("Conclusion: The Full Model performed better than the Golden Feature model.")